<a href="https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is CTR / Engagement Opportunity Scoring. I frame it primarily as a scoring and ranking task. The goal is to assign each page an opportunity score based on signals such as impressions, CTR, average position, sessions, engagement, content type, and freshness. The pages can then be ranked so a reviewer knows which pages deserve attention firs


In [ ]:
# Clone your FlyRank repository
!git clone https://github.com/Randaadad/FlyRank-AI.git

# Load the dataset
import pandas as pd

df = pd.read_csv(
    "/content/FlyRank-AI/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
print("\nOne row represents one content page.")

display(df.head())

Cloning into 'FlyRank-AI'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 118 (delta 32), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 1.84 MiB | 14.09 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Dataset shape: (30000, 44)

One row represents one content page.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy
My initial target is a CTR opportunity proxy rather than a causal outcome. I will compare a page's observed CTR with the typical CTR for pages in the same position tier. A page with enough impressions and a lower-than-expected CTR can receive a higher opportunity score. This is an observed, decision-support signal, not proof that changing the page will increase click

In [ ]:
# Keep pages with enough impressions
visible = df[df["impressions_90d"] >= 100].copy()

# Expected CTR for each position tier
expected_ctr = visible.groupby("position_tier")["ctr"].transform("mean")

# CTR opportunity gap
# Positive = CTR is below the typical CTR for its position tier
visible["ctr_gap"] = expected_ctr - visible["ctr"]

print("Number of pages:", len(df))
print("Pages with >=100 impressions:", len(visible))

display(
    visible[
        [
            "content_id",
            "position_tier",
            "impressions_90d",
            "ctr",
            "ctr_gap"
        ]
    ].head(10)
)

Number of pages: 30000
Pages with >=100 impressions: 22006


,content_id,position_tier,impressions_90d,ctr,ctr_gap
0,content_304f48230142,striking,3803,0.76,-0.504218
1,content_a1fb4e703a9e,page_3_5,15320,0.05,0.092359
2,content_9aa793d4d895,page_3_5,12581,0.09,0.052359
3,content_331d6c4de07b,page_1,11751,0.49,-0.135240
4,content_d99b7a2d90ca,page_3_5,19140,0.13,0.012359
5,content_d4084a4bc775,page_1,3970,0.03,0.324760
7,content_a63219c6e95a,page_3_5,1724,0.06,0.082359
8,content_5e6c160719bc,page_3_5,32574,0.09,0.052359
9,content_c27558df2b0c,page_1,1240,0.16,0.194760
10,content_d8ee6cc6d642,top_3,20919,1.55,-1.215872


## 3. Success metric

My main success metric will be Precision@K, especially Precision@20 or Precision@50. This matches the real decision because a reviewer has limited capacity and will inspect only the highest-ranked pages. A good scoring system should put a high proportion of genuine CTR or engagement opportunity pages near the top of the ranking.

In [ ]:
print("Primary success metric: Precision@20 or Precision@50")
print("Reason: the output is a ranked review queue.")


Primary success metric: Precision@20 or Precision@50
Reason: the output is a ranked review queue.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content page. Each row represents one pseudonymized content item and contains observed search and engagement signals such as impressions, clicks, CTR, average position, sessions, engagement, content type, and freshness. The decision is made at the page level: whether a page should be reviewed for a CTR or engagement opportunit

In [ ]:
print("Unit of analysis: one content page")
print("Each row represents one pseudonymized content item.")
print("Decision level: page-level CTR / engagement opportunity")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

Unit of analysis: one content page
Each row represents one pseudonymized content item.
Decision level: page-level CTR / engagement opportunity
Number of rows: 30000
Number of columns: 44


## 5. Why ML beats a fixed rule here

A fixed rule such as "CTR below 0.5% means opportunity" can be too simple because CTR depends strongly on position, impressions, content type, and other context. Two pages with the same CTR may have very different opportunities if they appear in different position tiers or have different levels of exposure. ML or a data-driven scoring approach can combine several signals and rank pages more consistently. I will still compare the approach with a simple baseline rule and only keep the more complex method if it improves the decision metric.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.